# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub pandas
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
df = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.client_hash_id,
    m.impressions,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS days_since_created,
    d.content_type
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()
df["days_since_created"] = df["days_since_created"].clip(lower=0)


df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["top_3", "page_1 (4-10)", "page_2 (11-20)", "deep (20+)"]
)
# Benchmark CTR per tier, taken directly from Signal 2 weighted results
benchmark_ctr = {
    "top_3": 0.388,
    "page_1 (4-10)": 0.325,
    "page_2 (11-20)": 0.316,
    "deep (20+)": 0.136,
}
df["benchmark_ctr_pct"] = df["position_tier"].map(benchmark_ctr).astype(float)
# The rule, in code
df["visible_flag"] = (df["impressions"] >= 200).astype(int)
df["weak_ctr_flag"] = (df["ctr_pct"] < df["benchmark_ctr_pct"]).astype(int)
df["score"] = df["visible_flag"] * df["weak_ctr_flag"] * df["impressions"].fillna(0)
df["reason_code"] = np.where(
    (df["visible_flag"] == 1) & (df["weak_ctr_flag"] == 1),
    "weak_ctr_for_position",
    "no_action"
)
df["action"] = np.where(df["score"] > 0, "review", "monitor")
import os
os.makedirs("work/outputs", exist_ok=True)
df_sorted = df.sort_values("score", ascending=False)
df_sorted.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Queue written: {len(df_sorted):,} rows")
print(f"Pages flagged for review: {(df_sorted['action']=='review').sum():,}")
df_sorted[["content_hash_id","position_tier","impressions","ctr_pct",
           "benchmark_ctr_pct","score","reason_code","action"]].head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue written: 331,437 rows
Pages flagged for review: 57,496


,content_hash_id,position_tier,impressions,ctr_pct,benchmark_ctr_pct,score,reason_code,action
312568,content_e8a52cf3d5988c07,page_2 (11-20),244931.0,0.273138,0.316,244931.0,weak_ctr_for_position,review
68406,content_0e03de7680314cd5,top_3,221310.0,0.325336,0.388,221310.0,weak_ctr_for_position,review
182485,content_44f34c0a90047651,top_3,212404.0,0.011299,0.388,212404.0,weak_ctr_for_position,review
68371,content_8d7d99f109e19aa2,top_3,203497.0,0.142017,0.388,203497.0,weak_ctr_for_position,review
147648,content_36e53e9c707674fc,deep (20+),194579.0,0.124371,0.136,194579.0,weak_ctr_for_position,review
273699,content_b99ea6861864dea5,page_1 (4-10),194337.0,0.185760,0.325,194337.0,weak_ctr_for_position,review
68401,content_4ffe18112a5642e3,top_3,186983.0,0.313397,0.388,186983.0,weak_ctr_for_position,review
107824,content_acbcc847f8996314,page_1 (4-10),170808.0,0.153389,0.325,170808.0,weak_ctr_for_position,review
3677,content_471d9cabce329a66,page_1 (4-10),164885.0,0.240167,0.325,164885.0,weak_ctr_for_position,review
307358,content_fd2117c2c6790e4b,page_1 (4-10),151166.0,0.269902,0.325,151166.0,weak_ctr_for_position,review


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I am using k-means clustering. I fits because this method does not need any answer key. Since clustering is a unsupervised learning and no one has already ranked the pages so K-means clustering is perfect toolkit. I am clustering impressions, ctr_pct, avg_position, days_since_created, content_type. We have to standardize every numeric feature before clustering because some feature's numeric value is huge and some are very low. So the features with larger value may dominate.

In [ ]:
# This is the evidence behind the "we must standardize" claim above.
features_preview = df[["impressions", "ctr_pct", "avg_position", "days_since_created"]]
print(features_preview.describe().T[["mean", "std", "min", "max"]])


                           mean          std  min       max
impressions         1587.986675  5431.337724  1.0  617124.0
ctr_pct                0.459397     3.775992  0.0     100.0
avg_position          15.992270    18.097575  0.0     309.0
days_since_created   207.806307   120.531290  0.0     494.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Though we are doing clustering which has no labeled dataset, we still spliting the dataset using client_id. We split them into two groups, find clusters on one group, then check if those same-shaped clusters show up in the other group.
I choose grouped by client_id because pages from same client are not independent (same industry, same writing style). If we split randomly page-by-page, a client's pages could end up half in "training" and half in "testing"


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped split: all of one client's pages stay on the same side
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, holdout_idx = next(gss.split(df, groups=df["client_hash_id"]))

df_train = df.iloc[train_idx].copy()
df_holdout = df.iloc[holdout_idx].copy()

print(f"Train: {len(df_train):,} pages from {df_train['client_hash_id'].nunique()} clients")
print(f"Holdout: {len(df_holdout):,} pages from {df_holdout['client_hash_id'].nunique()} clients")

# Sanity check: zero overlap in clients between the two sides
overlap = set(df_train['client_hash_id']) & set(df_holdout['client_hash_id'])
print(f"Clients appearing in both sides (should be 0): {len(overlap)}")

Train: 281,614 pages from 38 clients
Holdout: 49,823 pages from 17 clients
Clients appearing in both sides (should be 0): 0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.